# Notebook 4a — Individual Model Training & Tuning
## HMDA 2023 Loan Denial Prediction | OOM-Safe for MacBook Air M3 16 GB

---

### OOM Fixes Applied (from root cause analysis)

| Fix | Old Value | New Value | Memory Saved |
|:----|:---------|:---------|:-------------|
| `spark.driver.memory` | 12g | **6g** | ~6 GB OS headroom |
| `master` | `local[*]` (8 cores) | **`local[2]`** | 75% less concurrent task memory |
| RF | 9-combo TVS grid | **Direct fit** (no TVS) | Eliminates all RF model multiplication |
| GBT | 18-combo TVS grid | **Direct fit** (no TVS) | Eliminates all GBT model multiplication |
| TVS parallelism | 2 | **1** | 50% less peak model memory |
| MLP data | Full 6.1M rows | **10% sample (~600K)** | MLP doesn't support weights anyway |
| Model cleanup | Never freed | **Delete TVS after each model** |
| Notebook split | Single notebook | **4a (training) + 4b (ensembles)** = fresh JVM |

### Model Ladder

| # | Model | Family | Complexity |
|:--|:------|:-------|:-----------|
| 0 | Majority Baseline | None | O(1) |
| 1 | Naive Bayes | Probabilistic | Very Low |
| 2 | Logistic Regression | Linear | Low |
| 3 | Linear SVM | Linear (max-margin) | Low |
| 4 | Decision Tree | Non-linear | Medium |
| 5 | Random Forest | Bagging ensemble | Medium-High |
| 6 | GBT (PySpark) | Boosting ensemble | High |
| 7 | Multilayer Perceptron | Neural Network | High |

**Output**: Best hyperparameters + metrics saved to JSON/CSV; best model objects saved to disk.
**Next**: Notebook 4b loads these artifacts for ensembles & final evaluation.

In [1]:
# ============================================================
# Cell 1: Imports, SparkSession & MLflow Setup
# ============================================================
# OOM FIX: driver.memory 12g → 7g, local[*] → local[4],
#          maxResultSize 4g → 2g

# ─── Core Spark ───
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, ArrayType
from pyspark.ml.linalg import Vectors, VectorUDT, DenseVector

# ─── PySpark ML: All Available Classifiers ───
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier,
    NaiveBayes,
    LinearSVC,
    MultilayerPerceptronClassifier,
    FMClassifier,
)

# ─── PySpark ML: Evaluation & Tuning ───
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.ml.tuning import (
    ParamGridBuilder,
    TrainValidationSplit,
    CrossValidator,
)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline

# ─── Visualization & Utilities ───
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json, time, os, gc, warnings
warnings.filterwarnings("ignore")

# ─── MLflow for experiment tracking ───
try:
    import mlflow
    import mlflow.spark
    MLFLOW_AVAILABLE = True
    print("MLflow imported successfully")
except ImportError:
    MLFLOW_AVAILABLE = False
    print("MLflow not installed — continuing without tracking.")

# ─── Spark Session (OOM-SAFE CONFIG) ───
# WHY 7g? 16 GB total - 3.5 GB macOS - 1 GB Python - 0.5 GB JVM off-heap
#        = ~11 GB available. 7g leaves ~4 GB buffer for spikes.
# WHY local[4]? Halves concurrent task memory vs local[*] (8 cores).
#        Spark on M3 is often I/O-bound, so 4 cores ≈ same wall-clock.
spark = (SparkSession.builder
    .appName("HMDA_2023_ModelTraining_4a")
    .master("local[2]")                     # OOM FIX v2: was local[4], need fewer concurrent tasks
    .config("spark.driver.memory", "6g")    # OOM FIX v2: was 7g, need more OS headroom
    .config("spark.driver.maxResultSize", "2g")  # OOM FIX: was 4g
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.driver.extraJavaOptions",
            "-XX:+UseG1GC -XX:G1HeapRegionSize=16m "
            "-XX:InitiatingHeapOccupancyPercent=35")
    .config("spark.memory.fraction", "0.6")
    .config("spark.memory.storageFraction", "0.3")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | Cores: {spark.sparkContext.defaultParallelism}")

MLflow not installed — continuing without tracking.


26/03/01 18:50:06 WARN Utils: Your hostname, Adityas-Laptop.local resolves to a loopback address: 127.0.0.1; using 10.150.186.221 instead (on interface en0)
26/03/01 18:50:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 18:50:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.8 | Cores: 2


26/03/01 18:50:07 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# ============================================================
# Cell 2: Load Data & Initialize MLflow Experiment
# ============================================================

DATA_DIR  = "/Users/adi/Desktop/assignmt/project/data/processed"
MODEL_DIR = f"{DATA_DIR}/models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Load feature-engineered Parquet from Notebook 3
train_df = spark.read.parquet(f"{DATA_DIR}/train_features.parquet").cache()
test_df  = spark.read.parquet(f"{DATA_DIR}/test_features.parquet").cache()

train_count = train_df.count()
test_count  = test_df.count()
feature_dim = train_df.first()["features"].size

# Load metadata
with open(f"{DATA_DIR}/feature_metadata.json") as f:
    feature_meta = json.load(f)

print("=" * 70)
print("DATA LOADED")
print("=" * 70)
print(f"  Train: {train_count:,} rows x {feature_dim} features")
print(f"  Test:  {test_count:,} rows x {feature_dim} features")

# Label distribution
for name, df in [("Train", train_df), ("Test", test_df)]:
    counts = df.groupBy("label").count().orderBy("label").collect()
    total = sum(r["count"] for r in counts)
    print(f"  {name}: " + " | ".join(
        f"{'Originated' if r['label']==0 else 'Denied'}: {r['count']:,} ({r['count']/total*100:.1f}%)"
        for r in counts))

# ── MLflow Setup ──
if MLFLOW_AVAILABLE:
    mlflow.set_tracking_uri(f"file:///{DATA_DIR}/mlruns")
    experiment_name = "HMDA_2023_Model_Training"
    mlflow.set_experiment(experiment_name)
    experiment = mlflow.get_experiment_by_name(experiment_name)
    print(f"\nMLflow experiment: {experiment_name}")
    print(f"  Experiment ID: {experiment.experiment_id}")
else:
    print("\nMLflow not available — metrics will be stored in-memory only.")

26/03/01 18:50:14 WARN MemoryStore: Not enough space to cache rdd_8_4 in memory! (computed 452.1 MiB so far)
26/03/01 18:50:14 WARN BlockManager: Persisting block rdd_8_4 to disk instead.
26/03/01 18:50:16 WARN MemoryStore: Not enough space to cache rdd_8_4 in memory! (computed 452.1 MiB so far)
26/03/01 18:50:16 WARN MemoryStore: Not enough space to cache rdd_8_4 in memory! (computed 452.1 MiB so far)


DATA LOADED
  Train: 6,148,713 rows x 145 features
  Test:  1,533,840 rows x 145 features


26/03/01 18:50:21 WARN MemoryStore: Not enough space to cache rdd_8_4 in memory! (computed 452.1 MiB so far)


  Train: Originated: 4,552,842 (74.0%) | Denied: 1,595,871 (26.0%)
  Test: Originated: 1,136,777 (74.1%) | Denied: 397,063 (25.9%)

MLflow not available — metrics will be stored in-memory only.


## Section 1 — Class Imbalance: The Foundation of Everything

### The Core Interview Question: "How do you handle class imbalance?"

**What it is**: Our data is ~80% originated (label=0) and ~20% denied (label=1). A model predicting "always originated" gets 80% accuracy — but catches zero denials.

**Three main strategies** (know all three for interviews):

| Strategy | Mechanism | When to use | Trade-off |
|:---------|:---------|:-----------|:----------|
| **Class weights** | Multiply minority loss by `N_maj/N_min` | Large datasets (>10K minority) | Equivalent to oversampling, no synthetic data |
| **Oversampling (SMOTE)** | Generate synthetic minority samples via k-NN interpolation | Small datasets (<1K minority) | Can create unrealistic data points in sparse regions |
| **Undersampling** | Remove majority samples | Very large datasets, quick iteration | Throws away potentially useful data |

**Why we use class weights (not SMOTE)**:
With ~1.2M denied applications, we have **abundant minority samples**. SMOTE was designed for scenarios with hundreds of minority samples. Class weights achieve the same mathematical effect — scaling the gradient by class frequency — without creating synthetic data points.

**The math**:
```
weight_i = N_total / (2 * N_class_i)

For originated (80%): weight = N / (2 * 0.8N) = 0.625
For denied (20%):     weight = N / (2 * 0.2N) = 2.500

Ratio: 2.5 / 0.625 = 4x — denied samples count 4x more
```

**Interview follow-up: "Why 2 in the denominator?"**
This normalizes so that `sum(weights) = N`, keeping the effective sample size unchanged. Without the 2, total effective weight would double, which affects regularization strength.

In [3]:
# ============================================================
# Cell 4: Compute Class Weights & Prepare Weighted DataFrames
# ============================================================

label_counts = {row["label"]: row["count"]
                for row in train_df.groupBy("label").count().collect()}
n_orig   = label_counts[0.0]
n_denied = label_counts[1.0]
total    = n_orig + n_denied

w0 = total / (2.0 * n_orig)     # originated weight
w1 = total / (2.0 * n_denied)   # denied weight (higher)

print(f"Class weights: originated={w0:.4f}, denied={w1:.4f} (ratio {w1/w0:.1f}x)")

# Add weight column
train_w = train_df.withColumn(
    "weight", F.when(F.col("label") == 1.0, F.lit(w1)).otherwise(F.lit(w0))
).cache()
test_w = test_df.withColumn(
    "weight", F.when(F.col("label") == 1.0, F.lit(w1)).otherwise(F.lit(w0))
).cache()

# Unpersist the raw DFs — we only need the weighted versions from here
train_df.unpersist()
test_df.unpersist()

print("Weighted DataFrames cached. Raw DataFrames unpersisted to free memory.")
print(f"  train_w: {train_w.count():,} rows")
print(f"  test_w:  {test_w.count():,} rows")

# ── OOM FIX: Create 20% sample for hyperparameter tuning ──
# At 6.1M rows, optimal hyperparams at 1.2M are virtually identical.
# This reduces tuning memory by ~5x for RF/GBT.
train_sample = train_w.sample(fraction=0.1, seed=42).cache()
sample_count = train_sample.count()
print(f"\n  train_sample (10% for tuning): {sample_count:,} rows")

26/03/01 18:50:23 WARN MemoryStore: Not enough space to cache rdd_8_4 in memory! (computed 452.1 MiB so far)


Class weights: originated=0.6753, denied=1.9264 (ratio 2.9x)
Weighted DataFrames cached. Raw DataFrames unpersisted to free memory.


26/03/01 18:50:27 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 452.1 MiB so far)
26/03/01 18:50:27 WARN BlockManager: Persisting block rdd_74_4 to disk instead.
26/03/01 18:50:29 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 452.1 MiB so far)
26/03/01 18:50:29 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 452.1 MiB so far)


  train_w: 6,148,713 rows


  test_w:  1,533,840 rows


26/03/01 18:50:35 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 452.1 MiB so far)



  train_sample (10% for tuning): 615,028 rows


## Section 2 — Evaluation Framework & MLflow Integration

### Metrics Cheat Sheet (interview-ready)

| Metric | Formula | What it measures | HMDA relevance |
|:-------|:--------|:----------------|:---------------|
| **Accuracy** | (TP+TN) / Total | Overall correctness | Misleading: majority class dominates |
| **Precision** | TP / (TP+FP) | "Of predictions, how many are correct?" | High = fewer false denial flags |
| **Recall (Sensitivity)** | TP / (TP+FN) | "Of actual positives, how many caught?" | High = fewer missed denials |
| **F1** | 2PR / (P+R) | Harmonic mean of P and R | Balanced trade-off |
| **ROC-AUC** | Area under ROC curve | Ranking quality, all thresholds | Good general metric but insensitive to imbalance |
| **PR-AUC** | Area under Precision-Recall curve | Ranking quality for positive class | **Best for imbalanced data** — our primary metric |

### Why PR-AUC > ROC-AUC for imbalanced data

ROC uses FPR = FP/(FP+TN). When TN is huge (millions of originated loans), even many false positives barely move the FPR. A mediocre model gets ROC-AUC > 0.85.

PR-AUC uses Precision = TP/(TP+FP). Every false positive hurts directly. This **penalizes** models that exploit the majority class.

**Interview answer**: *"For imbalanced classification, I use PR-AUC as the primary metric because it focuses on the minority class performance, unlike ROC-AUC which is inflated by the large number of true negatives."*

### MLflow Integration

Every model training logs:
- **Parameters**: All hyperparameters
- **Metrics**: ROC-AUC, PR-AUC, F1, Denial Precision/Recall/F1, training time
- **Tags**: Model family, complexity tier
- **Artifacts**: Feature importance plots (where applicable)

This enables: experiment comparison, hyperparameter search analysis, and reproducibility.

In [4]:
# ============================================================
# Cell 6: Shared Evaluation Utilities + MLflow Logger
# ============================================================

# ── PySpark Evaluators ──
eval_roc  = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
eval_pr   = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR")
eval_f1   = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
eval_acc  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

# ── Global storage ──
ALL_RESULTS = {}
ALL_PREDICTIONS = {}  # Stores PANDAS DFs (sampled), not Spark DFs

# ── Storage for best hyperparams (saved to disk for NB4b) ──
BEST_PARAMS = {}


def evaluate_model(predictions, model_name, train_time=0.0, params=None,
                   tags=None, log_mlflow=True):
    """Evaluate a model, print results, store globally, and log to MLflow."""
    metrics = {}
    try:
        metrics["ROC-AUC"] = eval_roc.evaluate(predictions)
    except:
        metrics["ROC-AUC"] = 0.5
    try:
        metrics["PR-AUC"] = eval_pr.evaluate(predictions)
    except:
        metrics["PR-AUC"] = 0.0
    metrics["F1"]       = eval_f1.evaluate(predictions)
    metrics["Accuracy"] = eval_acc.evaluate(predictions)

    # Confusion matrix — compute in ONE pass
    cm = (predictions
        .select(
            F.col("label").cast("int").alias("actual"),
            F.col("prediction").cast("int").alias("pred"))
        .groupBy("actual", "pred").count()
        .collect()
    )
    cm_dict = {(r["actual"], r["pred"]): r["count"] for r in cm}
    tp = cm_dict.get((1, 1), 0)
    fp = cm_dict.get((0, 1), 0)
    fn = cm_dict.get((1, 0), 0)
    tn = cm_dict.get((0, 0), 0)

    d_prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
    d_rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
    d_f1   = 2*d_prec*d_rec/(d_prec+d_rec) if (d_prec+d_rec)>0 else 0.0

    metrics.update({
        "Denial_Precision": d_prec, "Denial_Recall": d_rec, "Denial_F1": d_f1,
        "Train_Time_s": train_time,
        "Confusion": {"TP": tp, "FP": fp, "FN": fn, "TN": tn}
    })

    ALL_RESULTS[model_name] = metrics

    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    print(f"  ROC-AUC:          {metrics['ROC-AUC']:.4f}")
    print(f"  PR-AUC:           {metrics['PR-AUC']:.4f}  *** primary metric")
    print(f"  F1 (weighted):    {metrics['F1']:.4f}")
    print(f"  Accuracy:         {metrics['Accuracy']:.4f}")
    print(f"  ------------------------------------")
    print(f"  Denial Precision: {d_prec:.4f}")
    print(f"  Denial Recall:    {d_rec:.4f}")
    print(f"  Denial F1:        {d_f1:.4f}")
    print(f"  ------------------------------------")
    print(f"  Confusion:  TN={tn:,}  FP={fp:,}")
    print(f"              FN={fn:,}  TP={tp:,}")
    print(f"  Time: {train_time:.1f}s")

    if log_mlflow and MLFLOW_AVAILABLE:
        with mlflow.start_run(run_name=model_name):
            if params:
                for k, v in params.items():
                    mlflow.log_param(k, v)
            for k, v in metrics.items():
                if k != "Confusion" and isinstance(v, (int, float)):
                    mlflow.log_metric(k.replace("-", "_"), v)
            if tags:
                for k, v in tags.items():
                    mlflow.set_tag(k, v)
            mlflow.set_tag("model_name", model_name)

    return metrics


def store_predictions_for_ensemble(predictions, model_name, sample_frac=0.05):
    """Sample predictions to pandas for ensemble diversity analysis."""
    pdf = (predictions
        .select("label", extract_p1("probability").alias("prob"))
        .sample(fraction=sample_frac, seed=42)
        .toPandas()
    )
    ALL_PREDICTIONS[model_name] = pdf
    print(f"  Stored {len(pdf):,} sampled predictions for ensemble analysis")


# Helper UDF
extract_p1 = F.udf(lambda v: float(v[1]) if v is not None and len(v) > 1 else 0.5, DoubleType())

print("Evaluation framework ready.")
print(f"MLflow logging: {'ENABLED' if MLFLOW_AVAILABLE else 'DISABLED'}")

Evaluation framework ready.
MLflow logging: DISABLED


## Model 0 — Majority-Class Baseline

### Interview concept: Why always start with a baseline?

A baseline answers: *"How well can I do with zero intelligence?"* Every ML model must **justify its complexity** by beating the baseline. If GBT gets 82% accuracy and the baseline gets 80%, the 2% lift might not be worth the training cost, debugging effort, and interpretability loss.

**Expected HMDA baseline**: ~80% accuracy (predicts all "originated"), 0% denial recall, 0.50 ROC-AUC, near-zero PR-AUC.

In [5]:
# ============================================================
# Cell 7: Model 0 — Majority-Class Baseline
# ============================================================

start = time.time()

baseline_preds = test_w.select("label").withColumn(
    "prediction", F.lit(0.0)
).withColumn(
    "rawPrediction", F.udf(lambda: Vectors.dense([1.0, 0.0]), VectorUDT())()
).withColumn(
    "probability", F.udf(lambda: Vectors.dense([1.0, 0.0]), VectorUDT())()
)

evaluate_model(baseline_preds, "M0_Baseline", time.time()-start,
               params={"strategy": "majority_class"},
               tags={"family": "baseline", "complexity": "none"})


  M0_Baseline
  ROC-AUC:          0.5000
  PR-AUC:           0.2589  *** primary metric
  F1 (weighted):    0.6309
  Accuracy:         0.7411
  ------------------------------------
  Denial Precision: 0.0000
  Denial Recall:    0.0000
  Denial F1:        0.0000
  ------------------------------------
  Confusion:  TN=1,136,777  FP=0
              FN=397,063  TP=0
  Time: 0.1s


{'ROC-AUC': 0.5,
 'PR-AUC': 0.25886859124810935,
 'F1': 0.6309411940736901,
 'Accuracy': 0.7411314087518907,
 'Denial_Precision': 0.0,
 'Denial_Recall': 0.0,
 'Denial_F1': 0.0,
 'Train_Time_s': 0.06079530715942383,
 'Confusion': {'TP': 0, 'FP': 0, 'FN': 397063, 'TN': 1136777}}

## Model 1 — Naive Bayes: The Simplest Probabilistic Classifier

### How it works (interview explanation)

Naive Bayes applies **Bayes' theorem** with a strong "naive" independence assumption:

```
P(denied | features) = P(features | denied) * P(denied) / P(features)
```

The "naive" part: it assumes each feature is **conditionally independent** given the class:
```
P(income, LTV, DTI | denied) = P(income | denied) * P(LTV | denied) * P(DTI | denied)
```

This is almost certainly **wrong** for HMDA — income and loan amount are correlated. But despite this incorrect assumption, Naive Bayes often works surprisingly well because:

1. **It only needs to rank correctly**, not calibrate probabilities perfectly. For classification, we only need P(denied|X) > P(originated|X), which is easier than getting exact probabilities.
2. **It's extremely fast** — just counting frequencies. O(N*d) training, O(d) prediction.
3. **It handles high-dimensional sparse data well** — each feature contributes independently, so 60 features don't cause overfitting.

### PySpark's Multinomial NB

PySpark's `NaiveBayes` supports `multinomial` (default, for counts/frequencies), `bernoulli` (binary features), and `gaussian` (continuous). We use **`gaussian`** because our StandardScaler output can have negative values, and `multinomial` requires non-negative features. Using MinMaxScaler to fix this would convert sparse vectors to dense — exploding memory ~10x and causing OOM on 6M rows. Gaussian NB models each feature as N(μ, σ) per class, which works well for our continuous financial features.

### When Naive Bayes works well vs poorly

| Works well | Works poorly |
|:-----------|:------------|
| Text classification (bag-of-words → truly ~independent) | Correlated features (income + loan amount) |
| High-dimensional sparse data | Low-dimensional dense data |
| Quick baseline, fast iteration | When precise probability calibration needed |
| When training data is very small | When you have abundant data (more complex models converge) |

### Expected on HMDA
NB should **underperform** tree-based models because:
- Features are correlated (income, LTV, DTI are all related)
- Non-linear decision boundaries exist (DTI thresholds)
- But it should comfortably beat the majority baseline

In [6]:
# ============================================================
# Cell 8: Model 1 — Naive Bayes
# ============================================================
# OOM FIX: parallelism=2 → 1, delete TVS after extracting best model

print("=" * 70)
print("MODEL 1: NAIVE BAYES")
print("=" * 70)

start = time.time()

nb = NaiveBayes(
    featuresCol="features", labelCol="label", weightCol="weight",
    modelType="gaussian", smoothing=1.0,
)

nb_grid = (ParamGridBuilder()
    .addGrid(nb.smoothing, [0.1, 0.5, 1.0, 2.0])
    .build()
)

nb_tvs = TrainValidationSplit(
    estimator=nb, estimatorParamMaps=nb_grid,
    evaluator=eval_pr, trainRatio=0.8, seed=42,
    parallelism=1,  # OOM FIX: was 2
)

print(f"Tuning {len(nb_grid)} smoothing values...")
nb_tvs_model = nb_tvs.fit(train_w)
nb_time = time.time() - start

best_nb = nb_tvs_model.bestModel
best_smoothing = best_nb._java_obj.getSmoothing()
BEST_PARAMS["NB"] = {"smoothing": best_smoothing}

print(f"  Best smoothing: {best_smoothing}")
for p, s in zip(nb_grid, nb_tvs_model.validationMetrics):
    print(f"    smoothing={p[nb.smoothing]:.1f} -> {s:.4f}")

nb_preds = nb_tvs_model.transform(test_w)
evaluate_model(nb_preds, "M1_NaiveBayes", nb_time,
    params={"smoothing": best_smoothing, "modelType": "gaussian"},
    tags={"family": "probabilistic", "complexity": "very_low"})

store_predictions_for_ensemble(nb_preds, "NaiveBayes")
nb_preds.unpersist()

# OOM FIX: Release TVS (holds all 4 candidate models)
del nb_tvs, nb_tvs_model
gc.collect()
print("  Freed TVS objects.")

MODEL 1: NAIVE BAYES
Tuning 4 smoothing values...


26/03/01 18:51:48 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 452.1 MiB so far)


  Best smoothing: 2.0
    smoothing=0.1 -> 0.9939
    smoothing=0.5 -> 0.9939
    smoothing=1.0 -> 0.9939
    smoothing=2.0 -> 0.9939


26/03/01 18:52:01 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/01 18:52:01 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS



  M1_NaiveBayes
  ROC-AUC:          0.9986
  PR-AUC:           0.9937  *** primary metric
  F1 (weighted):    0.9921
  Accuracy:         0.9921
  ------------------------------------
  Denial Precision: 0.9921
  Denial Recall:    0.9772
  Denial F1:        0.9846
  ------------------------------------
  Confusion:  TN=1,133,706  FP=3,071
              FN=9,061  TP=388,002
  Time: 61.5s


  Stored 76,384 sampled predictions for ensemble analysis
  Freed TVS objects.


## Model 2 — Logistic Regression: The Linear Workhorse

### How it works (mathematical intuition)

LR models the **log-odds** (logit) of the positive class as a linear function:

```
logit(P(denied)) = log(P(denied) / P(originated)) = w0 + w1*x1 + w2*x2 + ... + wd*xd
```

Then converts to probability via the **sigmoid** (logistic) function:
```
P(denied) = 1 / (1 + exp(-logit))
```

The sigmoid squashes any real number into [0, 1]:
- logit = 0 → P = 0.5 (maximum uncertainty)
- logit >> 0 → P → 1 (confident denial)
- logit << 0 → P → 0 (confident approval)

### Training: Maximum Likelihood via L-BFGS

LR finds weights that **maximize** the likelihood of the observed labels. The loss function is **binary cross-entropy** (log-loss):

```
L = -1/N * sum[ y_i * log(P_i) + (1 - y_i) * log(1 - P_i) ]
```

This is **convex** — guaranteed to have a single global minimum. L-BFGS (Limited-memory Broyden-Fletcher-Goldfarb-Shanno) is a quasi-Newton optimizer that finds it efficiently.

### Regularization (critical interview topic)

Without regularization, LR weights grow unbounded to perfectly separate training data → overfitting.

| Type | Penalty term | Effect | When to use |
|:-----|:------------|:-------|:-----------|
| **L2 (Ridge)** | `λ * sum(w²)` | Shrinks all weights toward zero; keeps all features | Correlated features (our OneHot-encoded cols) |
| **L1 (Lasso)** | `λ * sum(|w|)` | Drives some weights exactly to zero → feature selection | When you suspect many irrelevant features |
| **ElasticNet** | `α * L1 + (1-α) * L2` | Best of both: sparsity + grouping | Our best bet: sparse + correlated features |

### Why LR is the right FIRST "real" model for HMDA

1. **Interpretability**: Weight `w_i` directly tells you "a 1-unit increase in feature_i changes log-odds of denial by `w_i`". **Critical for fair lending** — regulators want to know which features drive denials.

2. **Strong baseline with good features**: Our NB3 feature engineering (log transforms, ratios, indicators) already linearized many relationships. LR exploits all of this.

3. **Fast**: Convex optimization on 6M rows converges in minutes. Trees take much longer.

### What LR cannot capture

- **Threshold effects**: DTI > 43% is a hard lending rule. LR models this as a smooth gradient, not a cliff.
- **Feature interactions**: If high LTV only matters when income is low, LR treats them independently.
- **Non-monotonic relationships**: If medium values behave differently from both low and high values.

In [7]:
# ============================================================
# Cell 9: Model 2 — Logistic Regression with Tuning
# ============================================================
# OOM FIX: parallelism=1, delete TVS after

print("=" * 70)
print("MODEL 2: LOGISTIC REGRESSION")
print("=" * 70)

start = time.time()

lr = LogisticRegression(
    featuresCol="features", labelCol="label", weightCol="weight",
    maxIter=100, family="binomial", standardization=False,
)

lr_grid = (ParamGridBuilder()
    .addGrid(lr.regParam, [0.001, 0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.3, 0.7, 1.0])
    .build()
)

lr_tvs = TrainValidationSplit(
    estimator=lr, estimatorParamMaps=lr_grid,
    evaluator=eval_pr, trainRatio=0.8, seed=42,
    parallelism=1,  # OOM FIX: was 2
)

print(f"Tuning {len(lr_grid)} combinations: regParam(3) x elasticNet(4)")
lr_tvs_model = lr_tvs.fit(train_w)
lr_time = time.time() - start

best_lr = lr_tvs_model.bestModel
best_rp = best_lr._java_obj.getRegParam()
best_en = best_lr._java_obj.getElasticNetParam()
BEST_PARAMS["LR"] = {"regParam": best_rp, "elasticNetParam": best_en}

print(f"\nBest: regParam={best_rp}, elasticNet={best_en}")
scored = sorted(zip(lr_grid, lr_tvs_model.validationMetrics), key=lambda x: -x[1])
for p, s in scored[:5]:
    print(f"  reg={p[lr.regParam]:.3f}, en={p[lr.elasticNetParam]:.1f} -> {s:.4f}")

lr_preds = lr_tvs_model.transform(test_w)
evaluate_model(lr_preds, "M2_LogisticRegression", lr_time,
    params={"regParam": best_rp, "elasticNetParam": best_en, "maxIter": 100},
    tags={"family": "linear", "complexity": "low"})

store_predictions_for_ensemble(lr_preds, "LR")
lr_preds.unpersist()

# Coefficient analysis
coeffs = best_lr.coefficients.toArray()
n_nonzero = sum(1 for c in coeffs if abs(c) > 1e-6)
print(f"\nCoefficients: {len(coeffs)} total, {n_nonzero} non-zero")
print(f"Intercept: {best_lr.intercept:.4f}")
top_idx = np.argsort(np.abs(coeffs))[::-1][:10]
print(f"Top 10 by |weight|:")
for rank, idx in enumerate(top_idx, 1):
    print(f"  {rank}. Feature[{idx}]: {coeffs[idx]:+.4f}")

# OOM FIX: Release TVS (12 candidate models)
del lr_tvs, lr_tvs_model
gc.collect()
print("  Freed TVS objects.")

MODEL 2: LOGISTIC REGRESSION
Tuning 12 combinations: regParam(3) x elasticNet(4)


26/03/01 18:52:14 WARN MemoryStore: Not enough space to cache rdd_444_2 in memory! (computed 708.1 MiB so far)
26/03/01 18:52:14 WARN BlockManager: Persisting block rdd_444_2 to disk instead.
26/03/01 18:59:57 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 452.1 MiB so far)
26/03/01 19:00:05 WARN MemoryStore: Not enough space to cache rdd_2191_3 in memory! (computed 419.2 MiB so far)
26/03/01 19:00:05 WARN BlockManager: Persisting block rdd_2191_3 to disk instead.
26/03/01 19:00:05 WARN MemoryStore: Not enough space to cache rdd_2191_2 in memory! (computed 630.5 MiB so far)
26/03/01 19:00:05 WARN BlockManager: Persisting block rdd_2191_2 to disk instead.



Best: regParam=0.001, elasticNet=0.0
  reg=0.001, en=0.0 -> 0.9989
  reg=0.001, en=0.3 -> 0.9988
  reg=0.001, en=0.7 -> 0.9984
  reg=0.010, en=0.0 -> 0.9981
  reg=0.001, en=1.0 -> 0.9977



  M2_LogisticRegression
  ROC-AUC:          0.9997
  PR-AUC:           0.9989  *** primary metric
  F1 (weighted):    0.9938
  Accuracy:         0.9938
  ------------------------------------
  Denial Precision: 0.9876
  Denial Recall:    0.9884
  Denial F1:        0.9880
  ------------------------------------
  Confusion:  TN=1,131,844  FP=4,933
              FN=4,592  TP=392,471
  Time: 499.6s


  Stored 76,384 sampled predictions for ensemble analysis

Coefficients: 145 total, 145 non-zero
Intercept: -1.6376
Top 10 by |weight|:
  1. Feature[16]: +4.5889
  2. Feature[26]: -0.3387
  3. Feature[102]: +0.3190
  4. Feature[105]: -0.2655
  5. Feature[25]: +0.2557
  6. Feature[100]: -0.2129
  7. Feature[107]: +0.2034
  8. Feature[129]: -0.1845
  9. Feature[128]: +0.1845
  10. Feature[2]: -0.1720
  Freed TVS objects.


## Model 3 — Support Vector Machine (Linear SVM): Maximum Margin Classification

### How SVM works (geometric intuition)

SVM finds the **hyperplane** that separates classes with the **maximum margin** — the widest possible gap between the nearest points of each class (called "support vectors").

```
                 Margin
          <───────────────>
 ○ ○      |     |     |      × ×
  ○ ○     | SV  |     | SV   × ×
   ○      |  ○  |     |  ×  ×
 ○  ○     |     |     |       ×
          ← - - - - - - - - →
           Decision Boundary
```

**Why maximum margin?** Statistical learning theory (Vapnik) proves that the hyperplane with the widest margin has the **best generalization guarantee** — it minimizes the VC dimension, reducing overfitting risk.

### The math (simplified)

Objective: minimize `||w||²/2` subject to `y_i * (w·x_i + b) >= 1` for all training points.

The `C` parameter (regParam in PySpark) controls the trade-off:
- **Large C**: Small margin, few misclassifications (hard margin → overfits)
- **Small C**: Large margin, allows misclassifications (soft margin → regularized)

### SVM vs Logistic Regression

| Aspect | SVM | LR |
|:-------|:----|:---|
| Objective | Maximize margin (hinge loss) | Maximize likelihood (log loss) |
| Decision boundary | Only depends on support vectors | Depends on all data points |
| Probability outputs | No (only raw scores) | Yes (calibrated probabilities) |
| Sparse solutions | Only support vectors matter | All points contribute to weights |
| With kernels | Can model non-linear boundaries | Stays linear (without kernel trick) |

### PySpark's LinearSVC limitations

PySpark only has **Linear** SVC — no kernel trick (no RBF, polynomial kernels). This means it's mathematically similar to LR but with hinge loss instead of log loss. For HMDA, we expect **similar performance to LR** because both are linear.

**Important**: LinearSVC does NOT output probability vectors — only `rawPrediction`. We'll handle this in evaluation.

### When SVM excels

- **High-dimensional sparse data** (text classification with thousands of features)
- **When margin matters more than probability** (binary decisions, not ranked lists)
- **Small to medium datasets** where the kernel trick captures non-linearity (not available in PySpark)

### Expected on HMDA

Similar to LR — both are linear. SVM might edge ahead slightly if the margin-maximizing objective is better suited to the feature space geometry, but the difference is usually small on well-engineered features.

In [8]:
# ============================================================
# Cell 10: Model 3 — Linear SVM
# ============================================================
# OOM FIX: parallelism=1, delete TVS after

print("=" * 70)
print("MODEL 3: LINEAR SVM")
print("=" * 70)

start = time.time()

svm = LinearSVC(
    featuresCol="features", labelCol="label",
    weightCol="weight",
    maxIter=100,
    standardization=False,
)

svm_grid = (ParamGridBuilder()
    .addGrid(svm.regParam, [0.001, 0.01, 0.1, 1.0])
    .build()
)

svm_tvs = TrainValidationSplit(
    estimator=svm, estimatorParamMaps=svm_grid,
    evaluator=eval_pr, trainRatio=0.8, seed=42,
    parallelism=1,  # OOM FIX: was 2
)

print(f"Tuning {len(svm_grid)} regParam values...")
try:
    svm_tvs_model = svm_tvs.fit(train_w)
    svm_time = time.time() - start

    best_svm = svm_tvs_model.bestModel
    best_svm_rp = best_svm._java_obj.getRegParam()
    BEST_PARAMS["SVM"] = {"regParam": best_svm_rp}
    print(f"Best regParam: {best_svm_rp}")

    svm_preds_raw = svm_tvs_model.transform(test_w)
    sigmoid_udf = F.udf(
        lambda v: Vectors.dense([1.0/(1.0+np.exp(v[0])), 1.0/(1.0+np.exp(-v[0]))]),
        VectorUDT()
    )
    svm_preds = svm_preds_raw.withColumn("probability", sigmoid_udf("rawPrediction"))

    evaluate_model(svm_preds, "M3_LinearSVM", svm_time,
        params={"regParam": best_svm_rp, "maxIter": 100},
        tags={"family": "linear", "complexity": "low"})

    store_predictions_for_ensemble(svm_preds, "SVM")
    svm_preds.unpersist()

    svm_coeffs = best_svm.coefficients.toArray()
    n_sv_nz = sum(1 for c in svm_coeffs if abs(c) > 1e-6)
    print(f"\nSVM coefficients: {len(svm_coeffs)} total, {n_sv_nz} non-zero")

except Exception as e:
    print(f"SVM training failed: {e}")
    print("Skipping SVM and continuing...")
    ALL_RESULTS["M3_LinearSVM"] = {
        "ROC-AUC": 0, "PR-AUC": 0, "F1": 0, "Accuracy": 0,
        "Denial_Precision": 0, "Denial_Recall": 0, "Denial_F1": 0,
        "Train_Time_s": 0, "Confusion": {"TP":0,"FP":0,"FN":0,"TN":0}
    }
finally:
    # OOM FIX: Always release TVS
    try: del svm_tvs, svm_tvs_model
    except: pass
    gc.collect()
    print("  Freed TVS objects.")

MODEL 3: LINEAR SVM
Tuning 4 regParam values...


26/03/01 19:00:51 WARN MemoryStore: Not enough space to cache rdd_2353_2 in memory! (computed 708.1 MiB so far)
26/03/01 19:00:51 WARN BlockManager: Persisting block rdd_2353_2 to disk instead.
26/03/01 19:02:04 WARN MemoryStore: Not enough space to cache rdd_2658_3 in memory! (computed 419.2 MiB so far)
26/03/01 19:02:04 WARN BlockManager: Persisting block rdd_2658_3 to disk instead.
26/03/01 19:02:04 WARN MemoryStore: Not enough space to cache rdd_2658_2 in memory! (computed 630.5 MiB so far)
26/03/01 19:02:04 WARN BlockManager: Persisting block rdd_2658_2 to disk instead.
26/03/01 19:04:46 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 452.1 MiB so far)
26/03/01 19:04:55 WARN MemoryStore: Not enough space to cache rdd_3384_3 in memory! (computed 419.2 MiB so far)
26/03/01 19:04:55 WARN BlockManager: Persisting block rdd_3384_3 to disk instead.
26/03/01 19:04:55 WARN MemoryStore: Not enough space to cache rdd_3384_2 in memory! (computed 630.5 MiB so far)
26

Best regParam: 0.001



  M3_LinearSVM
  ROC-AUC:          0.9994
  PR-AUC:           0.9976  *** primary metric
  F1 (weighted):    0.9943
  Accuracy:         0.9943
  ------------------------------------
  Denial Precision: 0.9931
  Denial Recall:    0.9850
  Denial F1:        0.9890
  ------------------------------------
  Confusion:  TN=1,134,078  FP=2,699
              FN=5,970  TP=391,093
  Time: 308.8s


  Stored 76,384 sampled predictions for ensemble analysis

SVM coefficients: 145 total, 142 non-zero
  Freed TVS objects.


## Model 4 — Decision Tree: If-Then Rules That Capture Thresholds

### How Decision Trees work (step by step)

1. **Start** with all training data in one node
2. **For each feature and threshold**, compute the impurity reduction from splitting
3. **Split** on the feature/threshold that gives the maximum impurity reduction
4. **Recurse** on each child node until a stopping condition (max depth, min samples, etc.)

### Impurity measures (interview question)

| Measure | Formula | Behavior |
|:--------|:--------|:---------|
| **Gini** | `1 - sum(p_i²)` | 0 when pure, max at 50/50. Computationally cheaper. |
| **Entropy** | `-sum(p_i * log(p_i))` | 0 when pure, max at 50/50. Slightly more sensitive to class probabilities. |
| **Misclassification** | `1 - max(p_i)` | Linear. Not differentiable. Rarely used for growing (only pruning). |

**Interview tip**: "Gini and Entropy produce nearly identical trees 95% of the time. Gini is default because it avoids the log computation."

### Why DTs are critical for HMDA

Lending has **hard rules**: DTI > 43% is a Qualified Mortgage boundary. A DT finds `DTI <= 43.0` as an exact split point. LR approximates this as a gradual slope — close but not exact.

### The fatal flaw: HIGH VARIANCE

A single DT changes dramatically with small data changes. This is because early splits determine the entire tree — one different sample can flip the root split, creating a completely different tree. This is why DTs are rarely used alone, but are the **building block** for RF (variance reduction) and GBT (bias reduction).

In [9]:
# ============================================================
# Cell 11: Model 4 — Decision Tree
# ============================================================
# OOM FIX: parallelism=1, delete TVS after

print("=" * 70)
print("MODEL 4: DECISION TREE")
print("=" * 70)

start = time.time()

dt = DecisionTreeClassifier(
    featuresCol="features", labelCol="label", weightCol="weight",
    impurity="gini", seed=42,
)

dt_grid = (ParamGridBuilder()
    .addGrid(dt.maxDepth, [5, 10, 15, 20])
    .addGrid(dt.minInstancesPerNode, [100, 500, 1000])
    .build()
)

dt_tvs = TrainValidationSplit(
    estimator=dt, estimatorParamMaps=dt_grid,
    evaluator=eval_pr, trainRatio=0.8, seed=42,
    parallelism=1,  # OOM FIX: was 2
)

print(f"Tuning {len(dt_grid)} combos: maxDepth(4) x minInstances(3)")
dt_tvs_model = dt_tvs.fit(train_w)
dt_time = time.time() - start

best_dt = dt_tvs_model.bestModel
BEST_PARAMS["DT"] = {
    "maxDepth": best_dt.depth,
    "minInstancesPerNode": best_dt._java_obj.getMinInstancesPerNode()
}
print(f"\nBest: depth={best_dt.depth}, nodes={best_dt.numNodes}")

dt_preds = dt_tvs_model.transform(test_w)
evaluate_model(dt_preds, "M4_DecisionTree", dt_time,
    params={"maxDepth": best_dt.depth, "numNodes": best_dt.numNodes,
            "minInstancesPerNode": best_dt._java_obj.getMinInstancesPerNode()},
    tags={"family": "tree", "complexity": "medium"})

store_predictions_for_ensemble(dt_preds, "DT")
dt_preds.unpersist()

dt_imp = best_dt.featureImportances.toArray()
top_dt = np.argsort(dt_imp)[::-1][:10]
print(f"\nTop 10 features (Gini decrease):")
for r, i in enumerate(top_dt, 1):
    print(f"  {r}. Feature[{i}] = {dt_imp[i]:.4f}")

# OOM FIX: Release TVS (12 DT candidate models)
del dt_tvs, dt_tvs_model
gc.collect()
print("  Freed TVS objects.")

MODEL 4: DECISION TREE
Tuning 12 combos: maxDepth(4) x minInstances(3)


26/03/01 19:06:28 WARN MemoryStore: Not enough space to cache rdd_3720_2 in memory! (computed 708.1 MiB so far)
26/03/01 19:06:28 WARN BlockManager: Persisting block rdd_3720_2 to disk instead.
26/03/01 19:06:43 WARN MemoryStore: Not enough space to cache rdd_3720_2 in memory! (computed 708.1 MiB so far)
26/03/01 19:07:15 WARN MemoryStore: Not enough space to cache rdd_3823_2 in memory! (computed 553.8 MiB so far)
26/03/01 19:07:15 WARN BlockManager: Persisting block rdd_3823_2 to disk instead.
26/03/01 19:07:38 WARN MemoryStore: Not enough space to cache rdd_3720_2 in memory! (computed 708.1 MiB so far)
26/03/01 19:08:05 WARN MemoryStore: Not enough space to cache rdd_3955_2 in memory! (computed 553.8 MiB so far)
26/03/01 19:08:05 WARN BlockManager: Persisting block rdd_3955_2 to disk instead.
26/03/01 19:08:15 WARN MemoryStore: Not enough space to cache rdd_3955_1 in memory! (computed 485.6 MiB so far)
26/03/01 19:08:33 WARN MemoryStore: Not enough space to cache rdd_4036_2 in memory


Best: depth=15, nodes=529



  M4_DecisionTree
  ROC-AUC:          0.9994
  PR-AUC:           0.9982  *** primary metric
  F1 (weighted):    0.9938
  Accuracy:         0.9938
  ------------------------------------
  Denial Precision: 0.9860
  Denial Recall:    0.9902
  Denial F1:        0.9881
  ------------------------------------
  Confusion:  TN=1,131,199  FP=5,578
              FN=3,893  TP=393,170
  Time: 439.3s


  Stored 76,384 sampled predictions for ensemble analysis

Top 10 features (Gini decrease):
  1. Feature[16] = 0.9458
  2. Feature[117] = 0.0477
  3. Feature[7] = 0.0021
  4. Feature[100] = 0.0006
  5. Feature[83] = 0.0004
  6. Feature[86] = 0.0004
  7. Feature[125] = 0.0003
  8. Feature[0] = 0.0003
  9. Feature[105] = 0.0002
  10. Feature[23] = 0.0002
  Freed TVS objects.


## Model 5 — Random Forest: Variance Reduction via Bagging

### The core idea: "Wisdom of Crowds"

Train many **independent** trees on different random subsets, then **average** their predictions. Individual tree noise cancels out; shared signal survives.

### Three sources of randomness in RF

1. **Bootstrap sampling** (Bagging): Each tree trains on a random sample with replacement. ~63.2% unique rows; ~36.8% left out (OOB = Out-of-Bag set for free validation).

2. **Random feature subsets**: At each split, only `sqrt(d)` randomly chosen features are considered. Forces trees to explore different splitting strategies.

3. **Different tree structures**: Combined effect of (1) + (2) = diverse trees that make different errors.

### The variance reduction math

If each of B trees has variance `sigma^2` and correlation `rho` between tree predictions:
```
Var(RF) = rho * sigma^2 + (1 - rho) * sigma^2 / B
```

- As B → infinity, the second term vanishes → variance = `rho * sigma^2`
- Random feature selection reduces `rho` → this is why RF beats simple bagging
- **Interview insight**: Adding more trees NEVER increases variance — it can only help (diminishing returns)

### RF hyperparameters

| Parameter | Effect | Too low | Too high |
|:----------|:-------|:--------|:---------|
| `numTrees` | Number of trees | Underfit (high variance) | Diminishing returns (slow) |
| `maxDepth` | Per-tree depth | Weak learners (high bias) | Overfit per tree (but ensemble corrects) |
| `featureSubsetStrategy` | Features per split | More diversity, weaker trees | Less diversity, stronger trees |
| `subsamplingRate` | Row sampling | More diversity | Less diversity (1.0 = full bootstrap) |

In [10]:
# ============================================================
# Cell 12: Model 5 — Random Forest
# ============================================================
# OOM FIX v2: TVS with RF killed the block manager even on 20% sample.
# maxDepth=14 → 2^14 = 16,384 nodes/tree × 200 trees × 4 combos = OOM.
#
# Solution: Skip TVS entirely. Train ONE RF with well-chosen params.
# At 6M+ rows, RF is not very sensitive to numTrees (100 vs 200 ≈ same)
# and maxDepth=10 captures all real lending thresholds.

print("=" * 70)
print("MODEL 5: RANDOM FOREST (direct fit — no grid search)")
print("=" * 70)

start = time.time()

# Fixed hyperparams — chosen from domain knowledge + ML best practices:
#   numTrees=150: sweet spot (100 is fine, 200 is marginal gain, 150 is middle)
#   maxDepth=10:  2^10 = 1,024 nodes/tree — captures DTI/LTV thresholds without OOM
#                 (depth 14 was 16x more nodes and killed the JVM)
best_rf = RandomForestClassifier(
    featuresCol="features", labelCol="label", weightCol="weight",
    numTrees=150, maxDepth=10,
    featureSubsetStrategy="sqrt", impurity="gini", seed=42,
).fit(train_w)
rf_time = time.time() - start

best_rf_trees = 150
best_rf_depth = 10
BEST_PARAMS["RF"] = {"numTrees": best_rf_trees, "maxDepth": best_rf_depth}

print(f"  Trained in {rf_time:.1f}s: {best_rf_trees} trees, depth {best_rf_depth}")

rf_preds = best_rf.transform(test_w)
evaluate_model(rf_preds, "M5_RandomForest", rf_time,
    params={"numTrees": best_rf_trees, "maxDepth": best_rf_depth},
    tags={"family": "bagging", "complexity": "medium_high"})

store_predictions_for_ensemble(rf_preds, "RF")
rf_preds.unpersist()

rf_imp = best_rf.featureImportances.toArray()
top_rf = np.argsort(rf_imp)[::-1][:10]
print(f"\nTop 10 features (avg Gini across {best_rf_trees} trees):")
for r, i in enumerate(top_rf, 1):
    print(f"  {r}. Feature[{i}] = {rf_imp[i]:.4f}")

gc.collect()

MODEL 5: RANDOM FOREST (direct fit — no grid search)


26/03/01 19:13:52 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 452.1 MiB so far)
26/03/01 19:13:54 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 452.1 MiB so far)
26/03/01 19:13:56 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 452.1 MiB so far)
26/03/01 19:14:03 WARN MemoryStore: Not enough space to cache rdd_4970_1 in memory! (computed 818.9 MiB so far)
26/03/01 19:14:03 WARN BlockManager: Persisting block rdd_4970_1 to disk instead.
26/03/01 19:14:26 WARN MemoryStore: Not enough space to cache rdd_4970_2 in memory! (computed 818.9 MiB so far)
26/03/01 19:14:26 WARN BlockManager: Persisting block rdd_4970_2 to disk instead.
26/03/01 19:14:26 WARN MemoryStore: Not enough space to cache rdd_4970_3 in memory! (computed 818.9 MiB so far)
26/03/01 19:14:26 WARN BlockManager: Persisting block rdd_4970_3 to disk instead.
26/03/01 19:14:44 WARN MemoryStore: Not enough space to cache rdd_4970_2 in memory! (com

  Trained in 871.4s: 150 trees, depth 10


26/03/01 19:28:21 WARN DAGScheduler: Broadcasting large task binary with size 5.8 MiB
26/03/01 19:28:46 WARN DAGScheduler: Broadcasting large task binary with size 5.8 MiB
26/03/01 19:29:13 WARN DAGScheduler: Broadcasting large task binary with size 5.8 MiB
26/03/01 19:29:38 WARN DAGScheduler: Broadcasting large task binary with size 5.8 MiB
26/03/01 19:30:02 WARN DAGScheduler: Broadcasting large task binary with size 5.8 MiB
26/03/01 19:30:25 WARN DAGScheduler: Broadcasting large task binary with size 5.8 MiB
26/03/01 19:30:25 WARN DAGScheduler: Broadcasting large task binary with size 5.8 MiB



  M5_RandomForest
  ROC-AUC:          0.9995
  PR-AUC:           0.9981  *** primary metric
  F1 (weighted):    0.9942
  Accuracy:         0.9942
  ------------------------------------
  Denial Precision: 0.9898
  Denial Recall:    0.9878
  Denial F1:        0.9888
  ------------------------------------
  Confusion:  TN=1,132,739  FP=4,038
              FN=4,863  TP=392,200
  Time: 871.4s


  Stored 76,384 sampled predictions for ensemble analysis

Top 10 features (avg Gini across 150 trees):
  1. Feature[16] = 0.5123
  2. Feature[2] = 0.2317
  3. Feature[105] = 0.0249
  4. Feature[144] = 0.0245
  5. Feature[43] = 0.0243
  6. Feature[102] = 0.0140
  7. Feature[25] = 0.0109
  8. Feature[44] = 0.0104
  9. Feature[7] = 0.0100
  10. Feature[26] = 0.0090


610

## Model 6 — Gradient Boosted Trees: The Tabular Data Champion

### Bagging vs Boosting: The Fundamental Split

| Property | Bagging (RF) | Boosting (GBT) |
|:---------|:------------|:---------------|
| Tree independence | Independent, parallel | Sequential, each corrects predecessors |
| What each tree sees | Random subset of data | Residual ERRORS from all prior trees |
| What it reduces | Variance | Bias |
| Overfit risk | Low (averaging is safe) | High (chasing errors → memorizing noise) |
| Training speed | Fast (parallel) | Slow (sequential) |

### How GBT works (step by step)

1. **Initialize** F₀(x) = log(P(denied)/P(originated)) — the log-odds base rate
2. For m = 1, 2, ..., M:
   a. Compute **pseudo-residuals**: r_i = y_i - P(y_i=1 | F_{m-1}(x_i))
      These are the "errors" — what the current model gets wrong
   b. Fit a shallow decision tree h_m(x) to predict residuals r_i
   c. Update: F_m(x) = F_{m-1}(x) + η * h_m(x)
      where η is the learning rate (shrinkage)
3. Final prediction: P(denied) = sigmoid(F_M(x))

**Key insight**: Each tree ONLY sees what previous trees got wrong. Tree #50 is specializing in the hardest edge cases — the marginal loans where approval vs denial depends on subtle feature combinations.

---

## The Complete Boosting Family: From AdaBoost to CatBoost

### AdaBoost (2003, Freund & Schapire)

**Core idea**: Re-weight training samples — give higher weights to misclassified points.

```
For each round m:
  1. Train weak learner on weighted data
  2. Compute weighted error ε_m
  3. Compute learner weight α_m = 0.5 * log((1 - ε_m) / ε_m)
  4. Update sample weights: increase for misclassified, decrease for correct
     w_i *= exp(α_m) if misclassified
     w_i *= exp(-α_m) if correct
```

**Limitation**: Only works with exponential loss. Sensitive to noisy labels (keeps boosting noise).
**NOT in PySpark ML** — but important to understand conceptually.

### Gradient Boosting (2001, Friedman) — What PySpark uses

**Improvement over AdaBoost**: Instead of re-weighting samples, fit each tree to the **gradient of the loss function** (residuals for squared error, pseudo-residuals for log-loss). This generalizes to ANY differentiable loss function.

**Why it's better**: AdaBoost is a special case of GB with exponential loss. GB with log-loss (deviance) is more robust to outliers and noisy labels.

### XGBoost (2016, Chen & Guestrin) — "Extreme Gradient Boosting"

**Improvements over vanilla GB**:
1. **Regularized objective**: Adds L1/L2 penalty on leaf weights → controls overfitting directly
   ```
   Obj = Loss + γ*T + 0.5*λ*sum(w²)   (T = #leaves, w = leaf weights)
   ```
2. **Newton's method (2nd-order gradients)**: Uses both gradient AND Hessian for better split finding → faster convergence
3. **Weighted quantile sketch**: Approximate split finding for distributed/large data
4. **Sparsity-aware**: Learns default direction for missing values → handles NaN natively
5. **Column block structure**: Cache-aware access pattern → 10x faster than sklearn GB

**Why XGBoost dominates Kaggle**: The regularized objective + 2nd-order optimization means it converges faster and overfits less than vanilla GB.

### LightGBM (2017, Microsoft)

**Improvements over XGBoost**:
1. **Leaf-wise growth** (vs XGBoost's level-wise):
   - XGBoost: Grows all leaves at the same depth → balanced trees
   - LightGBM: Grows the leaf with max loss reduction → unbalanced but more efficient
   - **Result**: Same accuracy in fewer leaves → 10-20x faster
2. **Gradient-based One-Side Sampling (GOSS)**: Keep all samples with large gradients (hard examples), randomly sample small-gradient (easy) ones → 2-5x speedup
3. **Exclusive Feature Bundling (EFB)**: Bundles mutually exclusive features (sparse OneHot columns) → reduces effective features
4. **Histogram-based splits**: Bins continuous features into 256 buckets → much faster split finding

**When to use over XGBoost**: Large datasets (>1M rows), many features, need fast iteration.

### CatBoost (2018, Yandex)

**Improvements**:
1. **Ordered boosting**: Prevents prediction shift (subtle overfitting in standard boosting where the same data is used for both gradient estimation and tree building)
2. **Native categorical support**: Handles categoricals WITHOUT OneHot encoding using ordered target statistics
3. **Symmetric trees**: All trees have the same structure → faster inference, better regularization

**When to use**: Heavy categorical features, production serving speed matters, fewer hyperparameters to tune.

### Comparison Summary

| Algorithm | Split finding | Growth | Missing values | Categoricals | Speed |
|:----------|:-------------|:-------|:---------------|:-------------|:------|
| Vanilla GB | Exact | Level-wise | Manual impute | Manual encode | Baseline |
| XGBoost | Approx quantile | Level-wise | Native default direction | Manual encode | 10x GB |
| LightGBM | Histogram | Leaf-wise | Native | Manual encode | 20x GB |
| CatBoost | Symmetric | Level-wise | Native | Native (ordered TS) | 15x GB |

**Interview answer**: *"I'd choose XGBoost as my default boosting algorithm for most tabular tasks. For very large datasets (>10M rows), I'd switch to LightGBM for speed. For heavy categorical data, CatBoost is ideal. PySpark's GBT is vanilla Gradient Boosting — solid but missing regularization and 2nd-order optimization."*

> **Note**: PySpark ML only includes vanilla GBTClassifier. For XGBoost/LightGBM/CatBoost in
> distributed Spark, you'd use `xgboost.spark`, `SynapseML`, or `catboost-spark` libraries.
> We train with PySpark's GBT here and explain the theory of all variants.

In [11]:
# ============================================================
# Cell 13: Model 6 — Gradient Boosted Trees
# ============================================================
# OOM FIX v2: Skip TVS for GBT too. Sequential training of 6 GBT
# candidates on even 10% sample risks the same block manager crash.
#
# GBT best practices for HMDA-scale tabular data:
#   maxIter=100-150, maxDepth=5-6, stepSize=0.1
# These are well-established; tuning gives <0.5% PR-AUC improvement.

print("=" * 70)
print("MODEL 6: GRADIENT BOOSTED TREES (direct fit — no grid search)")
print("=" * 70)

start = time.time()

# Fixed hyperparams — GBT literature + Spark defaults:
#   maxIter=120:  enough sequential trees for convergence
#   maxDepth=5:   standard for GBT (shallow trees, boosting compensates)
#   stepSize=0.1: standard learning rate
#   subsamplingRate=0.8: stochastic GBT reduces overfitting
best_gbt = GBTClassifier(
    featuresCol="features", labelCol="label", weightCol="weight",
    maxIter=120, maxDepth=5, stepSize=0.1,
    subsamplingRate=0.8, seed=42,
).fit(train_w)
gbt_time = time.time() - start

BEST_PARAMS["GBT"] = {
    "maxIter": 120, "maxDepth": 5,
    "stepSize": 0.1, "subsamplingRate": 0.8
}

print(f"  Trained in {gbt_time:.1f}s: 120 trees, depth 5, lr=0.1")

gbt_preds = best_gbt.transform(test_w)
evaluate_model(gbt_preds, "M6_GBT", gbt_time,
    params={"maxIter": 120, "maxDepth": 5,
            "stepSize": 0.1, "subsamplingRate": 0.8},
    tags={"family": "boosting", "complexity": "high"})

store_predictions_for_ensemble(gbt_preds, "GBT")
gbt_preds.unpersist()

gbt_imp = best_gbt.featureImportances.toArray()
top_gbt = np.argsort(gbt_imp)[::-1][:10]
print(f"\nTop 10 features (total gain across 120 trees):")
for r, i in enumerate(top_gbt, 1):
    print(f"  {r}. Feature[{i}] = {gbt_imp[i]:.4f}")

gc.collect()

MODEL 6: GRADIENT BOOSTED TREES (direct fit — no grid search)


26/03/01 19:30:54 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 452.1 MiB so far)
26/03/01 19:30:57 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 452.1 MiB so far)
26/03/01 19:31:08 WARN MemoryStore: Not enough space to cache rdd_5116_2 in memory! (computed 791.1 MiB so far)
26/03/01 19:31:08 WARN BlockManager: Persisting block rdd_5116_2 to disk instead.
26/03/01 19:31:08 WARN MemoryStore: Not enough space to cache rdd_5116_3 in memory! (computed 791.1 MiB so far)
26/03/01 19:31:08 WARN BlockManager: Persisting block rdd_5116_3 to disk instead.
26/03/01 19:31:14 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 708.1 MiB so far)
26/03/01 19:31:21 WARN MemoryStore: Not enough space to cache rdd_5116_3 in memory! (computed 527.4 MiB so far)
26/03/01 19:31:24 WARN MemoryStore: Not enough space to cache rdd_5116_3 in memory! (computed 527.4 MiB so far)
26/03/01 19:31:26 WARN MemoryStore: Not enough space to ca

  Trained in 3928.1s: 120 trees, depth 5, lr=0.1



  M6_GBT
  ROC-AUC:          0.9997
  PR-AUC:           0.9989  *** primary metric
  F1 (weighted):    0.9941
  Accuracy:         0.9941
  ------------------------------------
  Denial Precision: 0.9869
  Denial Recall:    0.9903
  Denial F1:        0.9886
  ------------------------------------
  Confusion:  TN=1,131,564  FP=5,213
              FN=3,844  TP=393,219
  Time: 3928.1s


  Stored 76,384 sampled predictions for ensemble analysis

Top 10 features (total gain across 120 trees):
  1. Feature[16] = 0.9447
  2. Feature[117] = 0.0140
  3. Feature[101] = 0.0133
  4. Feature[23] = 0.0062
  5. Feature[100] = 0.0028
  6. Feature[7] = 0.0018
  7. Feature[0] = 0.0017
  8. Feature[105] = 0.0016
  9. Feature[1] = 0.0012
  10. Feature[47] = 0.0012


781

## Model 7 — Multilayer Perceptron (MLP): Neural Network on Tabular Data

### How MLP works

An MLP is a feedforward neural network with:
- **Input layer**: One neuron per feature (60+ in our case)
- **Hidden layers**: Learned non-linear transformations
- **Output layer**: 2 neurons (originated, denied) with softmax

Each hidden neuron computes: `output = activation(W * input + b)`

**Activation functions**:
- **Sigmoid**: `1/(1+exp(-x))` — squashes to [0,1], but suffers from vanishing gradients
- **ReLU**: `max(0, x)` — default for hidden layers; fast, no vanishing gradient; but "dead neurons" possible
- **Softmax** (output layer): Converts raw scores to probabilities that sum to 1

### Backpropagation (training)

1. **Forward pass**: Input → hidden layers → output → loss
2. **Backward pass**: Compute gradient of loss w.r.t. each weight using chain rule
3. **Update weights**: `w_new = w_old - learning_rate * gradient`

### Why MLP is worth trying on HMDA

1. **Learns feature interactions automatically**: Unlike LR, an MLP with hidden layers can learn "income matters more when LTV is high" without explicit feature engineering
2. **Universal approximation theorem**: A single hidden layer with enough neurons can approximate ANY continuous function (given enough data and training)
3. **Non-linear representations**: Hidden layers create new "features" that may capture lending patterns trees miss

### Why MLP often loses to GBT on tabular data

1. **Requires more data per parameter**: MLP with 2 hidden layers of 64 neurons = ~4,000+ parameters to learn. GBT with 100 trees of depth 6 learns ~6,400 leaf values, but each is learned from local data subsets — more sample-efficient.
2. **No built-in handling of categorical/sparse data**: OneHot-encoded categoricals create high-dimensional sparse input — MLP treats each as a dense continuous feature.
3. **Hyperparameter sensitivity**: Hidden layer sizes, learning rate, batch size — many knobs vs GBT's 3-4 important ones.
4. **No feature importance**: MLP is a black box — bad for fair lending interpretability requirements.

### PySpark's MLP limitations

- Does NOT support `weightCol` → class weights not directly usable (imbalance handled less well)
- Fixed architecture (must specify layer sizes upfront)
- Uses L-BFGS optimizer (not SGD/Adam — less flexible)

In [12]:
# ============================================================
# Cell 14: Model 7 — Multilayer Perceptron
# ============================================================
# OOM FIX: Only try ONE architecture (smaller one) to save memory.
# MLP in PySpark doesn't support weightCol, and typically underperforms
# tree models on tabular data anyway.

print("=" * 70)
print("MODEL 7: MULTILAYER PERCEPTRON (NEURAL NETWORK)")
print("=" * 70)

start = time.time()

mlp_layers = [feature_dim, 64, 32, 2]  # input → 64 → 32 → output

print(f"  Training MLP: {mlp_layers}")
mlp = MultilayerPerceptronClassifier(
    featuresCol="features", labelCol="label",
    layers=mlp_layers,
    maxIter=100,
    blockSize=128,
    seed=42,
)

try:
    # MLP ignores weightCol — use weighted DF anyway (same distribution)
    mlp_model = mlp.fit(train_sample)  # OOM FIX v2: use 10% sample for MLP
    mlp_time = time.time() - start
    print(f"  Training completed in {mlp_time:.1f}s")

    mlp_preds = mlp_model.transform(test_w)  # evaluate on full test
    evaluate_model(mlp_preds, "M7_MLP", mlp_time,
        params={"layers": str(mlp_layers), "maxIter": 100, "blockSize": 128},
        tags={"family": "neural_network", "complexity": "high"})

    store_predictions_for_ensemble(mlp_preds, "MLP")
    mlp_preds.unpersist()
    del mlp_model
    gc.collect()

except Exception as e:
    print(f"  MLP training failed: {e}")
    print("  Skipping MLP and continuing...")
    ALL_RESULTS["M7_MLP"] = {
        "ROC-AUC": 0, "PR-AUC": 0, "F1": 0, "Accuracy": 0,
        "Denial_Precision": 0, "Denial_Recall": 0, "Denial_F1": 0,
        "Train_Time_s": 0, "Confusion": {"TP":0,"FP":0,"FN":0,"TN":0}
    }

MODEL 7: MULTILAYER PERCEPTRON (NEURAL NETWORK)
  Training MLP: [145, 64, 32, 2]


  Training completed in 248.5s



  M7_MLP
  ROC-AUC:          0.9894
  PR-AUC:           0.9846  *** primary metric
  F1 (weighted):    0.9936
  Accuracy:         0.9936
  ------------------------------------
  Denial Precision: 0.9924
  Denial Recall:    0.9827
  Denial F1:        0.9875
  ------------------------------------
  Confusion:  TN=1,133,790  FP=2,987
              FN=6,889  TP=390,174
  Time: 248.5s


  Stored 76,384 sampled predictions for ensemble analysis


In [13]:
# ============================================================
# Cell 15: Individual Model Leaderboard
# ============================================================

print("=" * 90)
print("INDIVIDUAL MODEL LEADERBOARD (sorted by PR-AUC)")
print("=" * 90)

rows = []
for name, m in ALL_RESULTS.items():
    rows.append({
        "Model": name,
        "ROC-AUC": m["ROC-AUC"],
        "PR-AUC": m["PR-AUC"],
        "Denial_F1": m["Denial_F1"],
        "Denial_Prec": m["Denial_Precision"],
        "Denial_Rec": m["Denial_Recall"],
        "Accuracy": m["Accuracy"],
        "Time_s": m["Train_Time_s"],
    })

leaderboard = pd.DataFrame(rows).sort_values("PR-AUC", ascending=False)
print(leaderboard.to_string(index=False, float_format="%.4f"))

print(f"\n>>> Current best: {leaderboard.iloc[0]['Model']} "
      f"(PR-AUC = {leaderboard.iloc[0]['PR-AUC']:.4f})")

INDIVIDUAL MODEL LEADERBOARD (sorted by PR-AUC)
                Model  ROC-AUC  PR-AUC  Denial_F1  Denial_Prec  Denial_Rec  Accuracy    Time_s
               M6_GBT   0.9997  0.9989     0.9886       0.9869      0.9903    0.9941 3928.1332
M2_LogisticRegression   0.9997  0.9989     0.9880       0.9876      0.9884    0.9938  499.5773
      M4_DecisionTree   0.9994  0.9982     0.9881       0.9860      0.9902    0.9938  439.3494
      M5_RandomForest   0.9995  0.9981     0.9888       0.9898      0.9878    0.9942  871.3715
         M3_LinearSVM   0.9994  0.9976     0.9890       0.9931      0.9850    0.9943  308.7870
        M1_NaiveBayes   0.9986  0.9937     0.9846       0.9921      0.9772    0.9921   61.4572
               M7_MLP   0.9894  0.9846     0.9875       0.9924      0.9827    0.9936  248.5354
          M0_Baseline   0.5000  0.2589     0.0000       0.0000      0.0000    0.7411    0.0608

>>> Current best: M6_GBT (PR-AUC = 0.9989)


In [14]:
# ============================================================
# Cell 16: Save Models, Params, Results for Notebook 4b
# ============================================================

print("=" * 70)
print("SAVING ARTIFACTS FOR NOTEBOOK 4b")
print("=" * 70)

# ── Save best model objects to disk ──
# Only save the 3 models needed for ensembles: LR, RF, GBT
try:
    best_lr = LogisticRegression(
        featuresCol="features", labelCol="label", weightCol="weight",
        maxIter=100, regParam=BEST_PARAMS["LR"]["regParam"],
        elasticNetParam=BEST_PARAMS["LR"]["elasticNetParam"],
        family="binomial", standardization=False,
    ).fit(train_w)
    best_lr.save(f"{MODEL_DIR}/best_lr")
    print(f"  Saved: {MODEL_DIR}/best_lr")
except Exception as e:
    print(f"  LR save skipped: {e}")

try:
    best_rf.save(f"{MODEL_DIR}/best_rf")
    print(f"  Saved: {MODEL_DIR}/best_rf")
except Exception as e:
    print(f"  RF save skipped: {e}")

try:
    best_gbt.save(f"{MODEL_DIR}/best_gbt")
    print(f"  Saved: {MODEL_DIR}/best_gbt")
except Exception as e:
    print(f"  GBT save skipped: {e}")

try:
    best_dt_model = DecisionTreeClassifier(
        featuresCol="features", labelCol="label", weightCol="weight",
        maxDepth=BEST_PARAMS["DT"]["maxDepth"],
        minInstancesPerNode=BEST_PARAMS["DT"]["minInstancesPerNode"],
        impurity="gini", seed=42,
    ).fit(train_w)
    best_dt_model.save(f"{MODEL_DIR}/best_dt")
    print(f"  Saved: {MODEL_DIR}/best_dt")
    del best_dt_model
except Exception as e:
    print(f"  DT save skipped: {e}")

# ── Save metrics ──
def sanitize(obj):
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, dict): return {k: sanitize(v) for k, v in obj.items()}
    return obj

results_path = f"{DATA_DIR}/model_results_4a.json"
with open(results_path, "w") as f:
    json.dump(sanitize(ALL_RESULTS), f, indent=2, default=str)
print(f"  Saved: {results_path}")

# ── Save best params ──
params_path = f"{DATA_DIR}/best_params_4a.json"
with open(params_path, "w") as f:
    json.dump(sanitize(BEST_PARAMS), f, indent=2, default=str)
print(f"  Saved: {params_path}")

# ── Save ensemble predictions (pandas) ──
preds_path = f"{DATA_DIR}/ensemble_predictions_4a.json"
preds_export = {k: v.to_dict(orient="list") for k, v in ALL_PREDICTIONS.items()}
with open(preds_path, "w") as f:
    json.dump(preds_export, f)
print(f"  Saved: {preds_path}")

# ── Save leaderboard ──
lb_path = f"{DATA_DIR}/leaderboard_4a.csv"
leaderboard.to_csv(lb_path, index=False)
print(f"  Saved: {lb_path}")

print(f"\nAll NB4a artifacts saved to {DATA_DIR}/")
print("\nNEXT: Run Notebook 4b for ensembles & final evaluation.")

SAVING ARTIFACTS FOR NOTEBOOK 4b


26/03/01 20:41:59 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 452.1 MiB so far)
26/03/01 20:42:12 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 708.1 MiB so far)


  Saved: /Users/adi/Desktop/assignmt/project/data/processed/models/best_lr


26/03/01 20:42:37 WARN TaskSetManager: Stage 4796 contains a task of very large size (2293 KiB). The maximum recommended task size is 1000 KiB.


  Saved: /Users/adi/Desktop/assignmt/project/data/processed/models/best_rf
  Saved: /Users/adi/Desktop/assignmt/project/data/processed/models/best_gbt


26/03/01 20:42:41 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 452.1 MiB so far)
26/03/01 20:42:44 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 452.1 MiB so far)
26/03/01 20:42:47 WARN MemoryStore: Not enough space to cache rdd_74_4 in memory! (computed 452.1 MiB so far)
26/03/01 20:42:59 WARN MemoryStore: Not enough space to cache rdd_8225_2 in memory! (computed 830.7 MiB so far)
26/03/01 20:42:59 WARN BlockManager: Persisting block rdd_8225_2 to disk instead.
26/03/01 20:42:59 WARN MemoryStore: Not enough space to cache rdd_8225_3 in memory! (computed 830.7 MiB so far)
26/03/01 20:42:59 WARN BlockManager: Persisting block rdd_8225_3 to disk instead.
26/03/01 20:43:11 WARN MemoryStore: Not enough space to cache rdd_8225_3 in memory! (computed 215.1 MiB so far)
26/03/01 20:43:14 WARN MemoryStore: Not enough space to cache rdd_8225_3 in memory! (computed 322.6 MiB so far)
26/03/01 20:43:16 WARN MemoryStore: Not enough space to ca

  Saved: /Users/adi/Desktop/assignmt/project/data/processed/models/best_dt
  Saved: /Users/adi/Desktop/assignmt/project/data/processed/model_results_4a.json
  Saved: /Users/adi/Desktop/assignmt/project/data/processed/best_params_4a.json
  Saved: /Users/adi/Desktop/assignmt/project/data/processed/ensemble_predictions_4a.json
  Saved: /Users/adi/Desktop/assignmt/project/data/processed/leaderboard_4a.csv

All NB4a artifacts saved to /Users/adi/Desktop/assignmt/project/data/processed/

NEXT: Run Notebook 4b for ensembles & final evaluation.


In [15]:
# ============================================================
# Cell 17: Stop Spark — Free JVM for Notebook 4b
# ============================================================
# OOM FIX: Stopping Spark here ensures NB4b gets a fresh JVM
# with clean memory. This is the single most impactful fix.

train_w.unpersist()
test_w.unpersist()
train_sample.unpersist()

spark.stop()
print("Spark stopped. JVM memory fully released.")
print("Run Notebook 4b for ensembles & evaluation.")

26/03/01 20:43:39 WARN BlockManager: Block rdd_74_2 could not be removed as it was not found on disk or in memory


Spark stopped. JVM memory fully released.
Run Notebook 4b for ensembles & evaluation.
